Trying to extract the most prominent k-mers from the reactive, bystander, self tolerant, and pre-immune datasets to compare what is the most prevalent marker of a tcr from that item

Imports

In [1]:
import polars as pl
import pandas as pd
import seaborn as sns
from pathlib import Path
import matplotlib.pyplot as plt

Path setting

In [2]:
data_dir_kb = Path("../Data/20250910 Comparison 2/Kb")
data_dir_kd = Path("../Data/20250910 Comparison 2/Kd")
output_dir = Path("Comparison2_v2")
output_dir.mkdir(exist_ok=True)

Reading kb files

In [3]:
kb_reactive = pl.concat([pl.read_csv(data_dir_kb / f"20250910 B10BR PD1hi{x} LL TCR Repertoire.csv") for x in 'ABCDE'])

kb_selfTolerant = pl.concat([pl.read_csv(data_dir_kb / "20250910 1783 Naive LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 1783 Naive SLO TCR Repertoire.csv")])

kb_bystander = pl.read_csv(data_dir_kb / "20250910 B10BR PD1negD LL TCR Repertoire.csv")

kb_preImmune = pl.concat([pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveA SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB LL TCR Repertoire.csv"), pl.read_csv(data_dir_kb / "20250910 B10BR NaiveB SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kb / "20250910 B10BR pre-immmune TCR Repertoire.csv")])

Reading kd files

In [4]:
kd_reactive = pl.concat([pl.read_csv(data_dir_kd / f"20251120 BL6 PD1hi{x} LL TCR Repertoire.csv") for x in 'ABC'])

kd_selfTolerant = pl.concat([pl.read_csv(data_dir_kd / "20250910 B6Kd Naive LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 B6Kd Naive SLO TCR Repertoire.csv")])

kd_bystander = pl.concat([pl.read_csv(data_dir_kd / "20250910 BL6 PD1negA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 PD1negB LL TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 BL6 PD1negC LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20251120 BL6 PD1negD LL TCR Repertoire.csv").drop('')])

kd_preImmune = pl.concat([pl.read_csv(data_dir_kd / "20250910 BL6 NaiveA LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 NaiveA SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 BL6 NaiveB LL TCR Repertoire.csv"), pl.read_csv(data_dir_kd / "20250910 BL6 NaiveB SLO TCR Repertoire.csv"),
                         pl.read_csv(data_dir_kd / "20250910 C57BL6 Pre-Immune SLO TCR Repertoire.csv")])

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kb trav

In [5]:
kb_reactive_trav_counts = kb_reactive['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (87,2)
kb_selfTolerant_trav_counts = kb_selfTolerant['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (143,2)
kb_preImmune_trav_counts = kb_preImmune['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (90,2)

kb_trav = (kb_reactive_trav_counts
            .join(kb_selfTolerant_trav_counts, on='TRAV', how='full', coalesce=True)
            .join(kb_preImmune_trav_counts, on='TRAV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kb_trav_freq = kb_trav.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kb_trav.write_csv(output_dir / 'kb_trav_counts')
kb_trav_freq.write_csv(output_dir / 'kb_trav_freqs')

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kb trbv

In [6]:
kb_reactive_trbv_counts = kb_reactive['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (23,2)
kb_selfTolerant_trbv_counts = kb_selfTolerant['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (25,2)
kb_preImmune_trbv_counts = kb_preImmune['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (25,2)

kb_trbv = (kb_reactive_trbv_counts
            .join(kb_selfTolerant_trbv_counts, on='TRBV', how='full', coalesce=True)
            .join(kb_preImmune_trbv_counts, on='TRBV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kb_trbv_freq = kb_trbv.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kb_trbv.write_csv(output_dir / 'kb_trbv_counts')
kb_trbv_freq.write_csv(output_dir / 'kb_trbv_freqs')

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kd trav

In [7]:
kd_reactive_trav_counts = kd_reactive['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (146,2)
kd_selfTolerant_trav_counts = kd_selfTolerant['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (123,2)
kd_preImmune_trav_counts = kd_preImmune['TRAV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (197,2)

kd_trav = (kd_reactive_trav_counts
            .join(kd_selfTolerant_trav_counts, on='TRAV', how='full', coalesce=True)
            .join(kd_preImmune_trav_counts, on='TRAV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kd_trav_freq = kd_trav.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kd_trav.write_csv(output_dir / 'kd_trav_counts')
kd_trav_freq.write_csv(output_dir / 'kd_trav_freqs')

Building dataframes that display the counts and freqencies of V-gene usage within the reactive, self-tolerant and preimmune datasets for kd trbv

In [8]:
kd_reactive_trbv_counts = kd_reactive['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_r'}) # (23,2)
kd_selfTolerant_trbv_counts = kd_selfTolerant['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_sT'}) # (24,2)
kd_preImmune_trbv_counts = kd_preImmune['TRBV'].value_counts().sort('count', descending=True).rename({'count': 'counts_pI'}) # (42,2)

kd_trbv = (kd_reactive_trbv_counts
            .join(kd_selfTolerant_trbv_counts, on='TRBV', how='full', coalesce=True)
            .join(kd_preImmune_trbv_counts, on='TRBV', how='full', coalesce=True)
            .with_columns(
                pl.col('^counts_.*$').fill_null(0)
            )
            .sort('counts_r', descending=True)
        )

kd_trbv_freq = kd_trbv.with_columns([
    pl.col('counts_r') / pl.col('counts_r').sum(),
    pl.col('counts_sT') / pl.col('counts_sT').sum(),
    pl.col('counts_pI') / pl.col('counts_pI').sum(),
])

kd_trbv.write_csv(output_dir / 'kd_trbv_counts')
kd_trbv_freq.write_csv(output_dir / 'kd_trbv_freqs')

Return all k-mers and their counts

In [9]:
def extract_kmers(df, col_name, out_alias, k):
    """Return all k-mers from an amino-acid sequence.
    Args:
        df - pl.DataFrame
        col_name - str,
        out_alias - str,
        k - int
    Returns:
        pl.DataFrame
    """
    return(
        df.select(
            pl.concat_list([
                pl.col(col_name).str.slice(i, k)
                for i in range(df[col_name].str.len_chars().max() - k + 1)
            ]).alias(f'{k}-mer')
        )
        .explode(f'{k}-mer', empty_as_null=True)
        .filter(pl.col(f'{k}-mer').str.len_chars() == k)
        .group_by(f'{k}-mer')
        .len(name=out_alias)
    )

one dataframe for everything x

In [10]:
def counts(df_list, col_name, aliases, k):
    """Merge all the counts into one table
    Args:
        df_list - list
        col_name - str
        aliases - list
        k - int
    Returns:
        pl.DataFrame"""
    basis = extract_kmers(df_list[0], col_name, aliases[0], k)
    for df, alias in zip(df_list[1:], aliases[1:]):
        counts = extract_kmers(df, col_name, alias, k)
        basis = basis.join(counts, on=f'{k}-mer', how='full', coalesce=True)
    return basis.fill_null(0)

markers

In [11]:
alpha = 'CDR3a_aa'
beta = 'CDR3b_aa'

defintion to subsample set by strata

In [12]:
def stratified_subsampling(reactive_counts, preImmune_df, gene):
    '''Takes a reactive and preimmune dataframe for a single category and
    sumsamples the preimmune dataframe down to the size and strata of the
    reactive dataset
    
    Args:
        reactive_counts - pl.DataFrame
        preImmune_df - pl.DataFrame
        gene - str
    Returns:
        stratified_pI - pl.DataFrame
    '''

    subsamples = []

    for row in reactive_counts.iter_rows(named=True):

        pI_pool = preImmune_df.filter(pl.col(gene) == row[gene])
        total = pI_pool.height
        sample_n = min(row['counts_r'], total)

        if sample_n > 0:
            taken = pI_pool.sample(n=sample_n, with_replacement=False, seed=1)
            subsamples.append(taken)

    stratified_pI = pl.concat(subsamples)

    return stratified_pI

Subsampling the preimmune datasets down to the size and proportions of the reactive sets

In [13]:
strat_kb_cdr3a_pI = stratified_subsampling(kb_reactive_trav_counts, kb_preImmune, 'TRAV')
strat_kb_cdr3b_pI = stratified_subsampling(kb_reactive_trbv_counts, kb_preImmune, 'TRBV')
strat_kd_cdr3a_pI = stratified_subsampling(kb_reactive_trav_counts, kd_preImmune, 'TRAV')
strat_kd_cdr3b_pI = stratified_subsampling(kd_reactive_trbv_counts, kd_preImmune, 'TRBV')

Trying out crude truncation of the CDR3 loops - getting rid of the first 3 and last 2 characters. This was informed by formation of the weblogos that showed that maximal conservation was in the first three characters and the final two characters of the string

In [14]:
chopped_kb_r = kb_reactive.with_columns([
    pl.col('CDR3a_aa').str.slice(3, pl.col('CDR3a_aa').str.len_chars() - 5),
    pl.col('CDR3b_aa').str.slice(3, pl.col('CDR3b_aa').str.len_chars() - 5)
    ])

chopped_kb_sT = kb_selfTolerant.with_columns([
    pl.col('CDR3a_aa').str.slice(3, pl.col('CDR3a_aa').str.len_chars() - 5),
    pl.col('CDR3b_aa').str.slice(3, pl.col('CDR3b_aa').str.len_chars() - 5)
    ])

chopped_kd_r = kd_reactive.with_columns([
    pl.col('CDR3a_aa').str.slice(3, pl.col('CDR3a_aa').str.len_chars() - 5),
    pl.col('CDR3b_aa').str.slice(3, pl.col('CDR3b_aa').str.len_chars() - 5)
    ])

chopped_kd_sT = kd_selfTolerant.with_columns([
    pl.col('CDR3a_aa').str.slice(3, pl.col('CDR3a_aa').str.len_chars() - 5),
    pl.col('CDR3b_aa').str.slice(3, pl.col('CDR3b_aa').str.len_chars() - 5)
    ])

Counting all 3-mers across the different datasets

In [15]:
kb_cdr3a_counts_df = counts(
    [kb_reactive, kb_selfTolerant, strat_kb_cdr3a_pI], alpha,
    ['kb_reactive_cdr3a', 'kb_selfTolerant_cdr3a', 'kb_preImmune_cdr3a'], 3
)

kb_cdr3b_counts_df = counts(
    [kb_reactive, kb_selfTolerant, strat_kb_cdr3b_pI], beta,
    ['kb_reactive_cdr3b', 'kb_selfTolerant_cdr3b', 'kb_preImmune_cdr3b'], 3
)

kd_cdr3a_counts_df = counts(
    [kd_reactive, kd_selfTolerant, strat_kd_cdr3a_pI], alpha,
    ['kd_reactive_cdr3a', 'kd_selfTolerant_cdr3a', 'kd_preImmune_cdr3a'], 3
)

kd_cdr3b_counts_df = counts(
    [kd_reactive, kd_selfTolerant, strat_kd_cdr3b_pI], beta,
    ['kd_reactive_cdr3b', 'kd_selfTolerant_cdr3b', 'kd_preImmune_cdr3b'], 3
)

kb_cdr3a_counts_df.write_csv(output_dir / 'kb_cdr3a_3mer_counts.csv')
kb_cdr3b_counts_df.write_csv(output_dir / 'kb_cdr3b_3mer_counts.csv')
kd_cdr3a_counts_df.write_csv(output_dir / 'kd_cdr3a_3mer_counts.csv')
kd_cdr3b_counts_df.write_csv(output_dir / 'kd_cdr3b_3mer_counts.csv')

In [16]:
chopped_kb_cdr3a_counts_df = counts(
    [chopped_kb_r, chopped_kb_sT, strat_kb_cdr3a_pI], alpha,
    ['kb_reactive_cdr3a', 'kb_selfTolerant_cdr3a', 'kb_preImmune_cdr3a'], 3
)

chopped_kb_cdr3b_counts_df = counts(
    [chopped_kb_r, chopped_kb_sT, strat_kb_cdr3b_pI], beta,
    ['kb_reactive_cdr3b', 'kb_selfTolerant_cdr3b', 'kb_preImmune_cdr3b'], 3
)

chopped_kd_cdr3a_counts_df = counts(
    [chopped_kd_r, chopped_kd_sT, strat_kd_cdr3a_pI], alpha,
    ['kd_reactive_cdr3a', 'kd_selfTolerant_cdr3a', 'kd_preImmune_cdr3a'], 3
)

chopped_kd_cdr3b_counts_df = counts(
    [chopped_kd_r, chopped_kd_sT, strat_kd_cdr3b_pI], beta,
    ['kd_reactive_cdr3b', 'kd_selfTolerant_cdr3b', 'kd_preImmune_cdr3b'], 3
)

chopped_kb_cdr3a_counts_df.write_csv(output_dir / 'chopped_kb_cdr3a_3mer_counts.csv')
chopped_kb_cdr3b_counts_df.write_csv(output_dir / 'chopped_kb_cdr3b_3mer_counts.csv')
chopped_kd_cdr3a_counts_df.write_csv(output_dir / 'chopped_kd_cdr3a_3mer_counts.csv')
chopped_kd_cdr3b_counts_df.write_csv(output_dir / 'chopped_kd_cdr3b_3mer_counts.csv')

Z scores of the 3-mers, relative to the row average - i.e. does this 3-mer appear more or less often than the average amount

Fold change of the 3-mer relative to the pre-immune dataset, expecting the key 3-mers to be up-regulated in the reactive and down-regulated in the selfTolerant

In [17]:
def freq_and_z(df, k):
    """Computes the frequencies of all the k-mers and then calculates the relative z-scores to compare between
        reactive, self tolerant, and pre immune datasets
    Args:
        df -> pl.DataFrame
    Returns:
        tuple[pl.DataFrame, pl.DataFrame]"""
    cols = [c for c in df.columns if c != f'{k}-mer']

    freq_df = df.with_columns([pl.col(c) / pl.col(c).sum() for c in cols])
    row_mean = pl.mean_horizontal(cols)
    variance_sum = pl.sum_horizontal([(pl.col(c) - row_mean) ** 2 for c in cols])
    row_std = (variance_sum / (len(cols) - 1)).sqrt()

    zscore_df = freq_df.with_columns([
        ((pl.col(c) - row_mean) / row_std).alias(c) for c in cols
    ])

    return freq_df, zscore_df

In [18]:
kb_cdr3a_freq_df, kb_cdr3a_z_df = freq_and_z(kb_cdr3a_counts_df, 3)
kb_cdr3b_freq_df, kb_cdr3b_z_df = freq_and_z(kb_cdr3b_counts_df, 3)
kd_cdr3a_freq_df, kd_cdr3a_z_df = freq_and_z(kd_cdr3a_counts_df, 3)
kd_cdr3b_freq_df, kd_cdr3b_z_df = freq_and_z(kd_cdr3b_counts_df, 3)

In [19]:
chopped_kb_cdr3a_freq_df, chopped_kb_cdr3a_z_df = freq_and_z(chopped_kb_cdr3a_counts_df, 3)
chopped_kb_cdr3b_freq_df, chopped_kb_cdr3b_z_df = freq_and_z(chopped_kb_cdr3b_counts_df, 3)
chopped_kd_cdr3a_freq_df, chopped_kd_cdr3a_z_df = freq_and_z(chopped_kd_cdr3a_counts_df, 3)
chopped_kd_cdr3b_freq_df, chopped_kd_cdr3b_z_df = freq_and_z(chopped_kd_cdr3b_counts_df, 3)

Making a heatmap to visualise the z score - picked the 3-mers with the highest variance

In [20]:
datasets = [
    ('kb_cdr3a', kb_cdr3a_freq_df, kb_cdr3a_z_df),
    ('kb_cdr3b', kb_cdr3b_freq_df, kb_cdr3b_z_df),
    ('kd_cdr3a', kd_cdr3a_freq_df, kd_cdr3a_z_df),
    ('kd_cdr3b', kd_cdr3b_freq_df, kd_cdr3b_z_df)
]

for name, freq_df, z_df in datasets:
    cols = [c for c in freq_df.columns if c != '3-mer']

    most_variable = (freq_df.select([
        pl.col('3-mer'),
        (pl.sum_horizontal([pl.col(c) - pl.mean_horizontal(cols) ** 2 for c in cols]) / (len(cols) - 1)).alias('var')
        ])
        .sort('var', descending=True)
        .head(500)
        .join(z_df, on='3-mer', how='inner')
        .drop('var')
        )
    
    labels = most_variable['3-mer'].to_numpy()
    numbers = most_variable.select(cols).to_numpy()
    
    sns_df = pd.DataFrame(numbers, columns=cols, index=labels)

    sns.clustermap(sns_df, cmap='vlag', center=0, figsize=(10,70), yticklabels=True)
    plt.savefig(output_dir / f'{name}_zscore_heatmap.png')
    plt.close()

In [21]:
chopped_datasets = [
    ('kb_cdr3a', chopped_kb_cdr3a_freq_df, chopped_kb_cdr3a_z_df),
    ('kb_cdr3b', chopped_kb_cdr3b_freq_df, chopped_kb_cdr3b_z_df),
    ('kd_cdr3a', chopped_kd_cdr3a_freq_df, chopped_kd_cdr3a_z_df),
    ('kd_cdr3b', chopped_kd_cdr3b_freq_df, chopped_kd_cdr3b_z_df)
]

for name, freq_df, z_df in datasets:
    cols = [c for c in freq_df.columns if c != '3-mer']

    most_variable = (freq_df.select([
        pl.col('3-mer'),
        (pl.sum_horizontal([pl.col(c) - pl.mean_horizontal(cols) ** 2 for c in cols]) / (len(cols) - 1)).alias('var')
        ])
        .sort('var', descending=True)
        .head(500)
        .join(z_df, on='3-mer', how='inner')
        .drop('var')
        )
    
    labels = most_variable['3-mer'].to_numpy()
    numbers = most_variable.select(cols).to_numpy()
    
    sns_df = pd.DataFrame(numbers, columns=cols, index=labels)

    sns.clustermap(sns_df, cmap='vlag', center=0, figsize=(10,70), yticklabels=True)
    plt.savefig(output_dir / f'chopped_{name}_zscore_heatmap.png')
    plt.close()